In [ ]:
import os
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from lib import PairedDataset, collate_paired_sequences
from lib import Trainer
from lib import ProteinInteractionModel

In [ ]:
spe = "yeast"

data_dir = "data"

input_dim = 13
hidden_dim = 25
batch_size = 32
n_epochs = 50


In [ ]:
train_file = os.path.join(data_dir, spe, "action/train_action.tsv")
val_file = os.path.join(data_dir, spe, "action/test_action.tsv")
embedding_h5 = os.path.join(data_dir, spe, "seq/pipr.embedding.h5")


train_df = pd.read_csv(train_file, sep="\t", header=None)
train_dataset = PairedDataset(train_df[0].to_list(), train_df[1].to_list(), 
                              train_df[2].to_list(), embedding_h5)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_paired_sequences,
    shuffle=True,
)

val_df = pd.read_csv(val_file, sep="\t", header=None)
val_dataset = PairedDataset(val_df[0].to_list(), val_df[1].to_list(), 
                              val_df[2].to_list(), embedding_h5)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=collate_paired_sequences
)

In [ ]:
model = ProteinInteractionModel(input_dim, hidden_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()

trainer = Trainer(model, loss_fn, optimizer)
trainer.set_loaders(train_loader, val_loader)
trainer.train(n_epochs)

In [ ]:
fig = trainer.plot_losses()